# Notebook 03: Disease Subgraph Network Analysis

Characterizing the structural properties of the acute brain injury disease subgraph in DRKG. This connects to Dr. Cheng's research in statistical network analysis and provides context for interpreting the drug repurposing results.

## Setup

In [1]:
import pandas as pd
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
import sys
sys.path.insert(1, '../utils')
from utils import download_and_extract

download_and_extract()

## Load DRKG triplets

In [2]:
df = pd.read_csv('../data/drkg.tsv', sep='\t', header=None, 
                  names=['source', 'relation', 'target'])

brain_injury_disease_list = [
    'Disease::MESH:D020521',  # Stroke
    'Disease::MESH:D002544',  # Cerebral Infarction
    'Disease::MESH:D020300',  # Intracranial Hemorrhages
    'Disease::MESH:D020520',  # Intracranial Hemorrhage, Hypertensive
    'Disease::MESH:D002538',  # Cerebral Hemorrhage
    'Disease::MESH:D001930',  # Brain Injuries
    'Disease::MESH:D006470',  # Hemorrhage
]

## Extract disease subgraph

Pull all triplets involving any of the target disease nodes.

In [3]:
disease_triplets = df[
    df['source'].isin(brain_injury_disease_list) | 
    df['target'].isin(brain_injury_disease_list)
]

print(f"Total triplets involving target diseases: {len(disease_triplets):,}")
print(f"\nRelation type breakdown:")
print(disease_triplets['relation'].value_counts().head(15))

Total triplets involving target diseases: 1,440

Relation type breakdown:
relation
GNBR::T::Compound:Disease             669
GNBR::J::Gene:Disease                 315
GNBR::L::Gene:Disease                 134
GNBR::Sa::Compound:Disease            106
DRUGBANK::treats::Compound:Disease     43
GNBR::Y::Gene:Disease                  37
GNBR::Te::Gene:Disease                 34
GNBR::X::Gene:Disease                  20
GNBR::U::Gene:Disease                  19
GNBR::D::Gene:Disease                  14
GNBR::Md::Gene:Disease                 14
GNBR::Pa::Compound:Disease             13
GNBR::G::Gene:Disease                  11
GNBR::J::Compound:Disease               6
GNBR::Pr::Compound:Disease              3
Name: count, dtype: int64


## Compound-disease connections

How many unique drugs connect to each disease node, and via which relation types.

In [4]:
compound_disease = disease_triplets[
    (disease_triplets['source'].str.startswith('Compound::') & 
     disease_triplets['target'].isin(brain_injury_disease_list)) |
    (disease_triplets['source'].isin(brain_injury_disease_list) & 
     disease_triplets['target'].str.startswith('Compound::'))
]

disease_labels = {
    'Disease::MESH:D020521': 'Stroke',
    'Disease::MESH:D002544': 'Cerebral Infarction',
    'Disease::MESH:D020300': 'Intracranial Hemorrhages',
    'Disease::MESH:D020520': 'ICH Hypertensive',
    'Disease::MESH:D002538': 'Cerebral Hemorrhage',
    'Disease::MESH:D001930': 'Brain Injuries',
    'Disease::MESH:D006470': 'Hemorrhage',
}

print("Unique compounds connected to each disease node:\n")
for disease, label in disease_labels.items():
    connected = set(
        compound_disease[compound_disease['target'] == disease]['source'].tolist() +
        compound_disease[compound_disease['source'] == disease]['target'].tolist()
    )
    print(f"  {label}: {len(connected)} compounds")

Unique compounds connected to each disease node:

  Stroke: 259 compounds
  Cerebral Infarction: 134 compounds
  Intracranial Hemorrhages: 14 compounds
  ICH Hypertensive: 7 compounds
  Cerebral Hemorrhage: 3 compounds
  Brain Injuries: 142 compounds
  Hemorrhage: 272 compounds


## Degree analysis

Total degree of each disease node in the full graph.

In [5]:
print("Total triplets per disease node:\n")
for disease, label in disease_labels.items():
    degree = ((df['source'] == disease) | (df['target'] == disease)).sum()
    print(f"  {label}: {degree:,} triplets")

Total triplets per disease node:

  Stroke: 442 triplets
  Cerebral Infarction: 273 triplets
  Intracranial Hemorrhages: 21 triplets
  ICH Hypertensive: 11 triplets
  Cerebral Hemorrhage: 7 triplets
  Brain Injuries: 290 triplets
  Hemorrhage: 396 triplets


## Gene connections

Genes connecting to brain injury disease nodes — these represent mechanistic intermediaries relevant to the explainability layer.

In [6]:
gene_disease = disease_triplets[
    (disease_triplets['source'].str.startswith('Gene::') & 
     disease_triplets['target'].isin(brain_injury_disease_list)) |
    (disease_triplets['source'].isin(brain_injury_disease_list) & 
     disease_triplets['target'].str.startswith('Gene::'))
]

print(f"Gene-disease triplets: {len(gene_disease):,}")

gene_counts = {}
for disease, label in disease_labels.items():
    genes = set(
        gene_disease[gene_disease['target'] == disease]['source'].tolist() +
        gene_disease[gene_disease['source'] == disease]['target'].tolist()
    )
    gene_counts[label] = len(genes)
    print(f"  {label}: {len(genes)} connected genes")

Gene-disease triplets: 598
  Stroke: 176 connected genes
  Cerebral Infarction: 138 connected genes
  Intracranial Hemorrhages: 7 connected genes
  ICH Hypertensive: 4 connected genes
  Cerebral Hemorrhage: 4 connected genes
  Brain Injuries: 147 connected genes
  Hemorrhage: 117 connected genes


## Disease-disease connections

Which of our target disease nodes are connected to each other in DRKG.

In [7]:
disease_disease = disease_triplets[
    disease_triplets['source'].isin(brain_injury_disease_list) & 
    disease_triplets['target'].isin(brain_injury_disease_list)
]

print(f"Disease-disease connections among target nodes: {len(disease_disease)}")
if len(disease_disease) > 0:
    print(disease_disease[['source', 'relation', 'target']])

Disease-disease connections among target nodes: 0


## Summary table

In [8]:
summary = []
for disease, label in disease_labels.items():
    total = ((df['source'] == disease) | (df['target'] == disease)).sum()
    compounds = len(set(
        compound_disease[compound_disease['target'] == disease]['source'].tolist() +
        compound_disease[compound_disease['source'] == disease]['target'].tolist()
    ))
    genes = gene_counts[label]
    summary.append({
        'disease': label,
        'mesh_id': disease,
        'total_triplets': total,
        'connected_compounds': compounds,
        'connected_genes': genes
    })

summary_df = pd.DataFrame(summary)
summary_df

,disease,mesh_id,total_triplets,connected_compounds,connected_genes
0,Stroke,Disease::MESH:D020521,442,259,176
1,Cerebral Infarction,Disease::MESH:D002544,273,134,138
2,Intracranial Hemorrhages,Disease::MESH:D020300,21,14,7
3,ICH Hypertensive,Disease::MESH:D020520,11,7,4
4,Cerebral Hemorrhage,Disease::MESH:D002538,7,3,4
5,Brain Injuries,Disease::MESH:D001930,290,142,147
6,Hemorrhage,Disease::MESH:D006470,396,272,117


## Save

In [9]:
import os
os.makedirs('../results', exist_ok=True)
summary_df.to_csv('../results/disease_subgraph_summary.csv', index=False)